In [2]:
import pandas as pd
import sqlite3
conn=sqlite3.connect('../data/olist_ecommerce.db')
print("connected to db successfully")


connected to db successfully


In [3]:
tables=pd.read_sql("""
    SELECT name
    FROM sqlite_master
    WHERE type='table'
""",conn)
print("table in our database")
print(tables)

table_names=tables['name'].tolist()

for table in table_names:
    count=pd.read_sql(f"SELECT COUNT(*) AS total_records FROM {table}",conn)
    print(f"{table:45}--> {count['total_records'][0]:,} rows")


table in our database
                                name
0            olist_customers_dataset
1          olist_geolocation_dataset
2               olist_orders_dataset
3          olist_order_items_dataset
4       olist_order_payments_dataset
5        olist_order_reviews_dataset
6             olist_products_dataset
7              olist_sellers_dataset
8  product_category_name_translation
olist_customers_dataset                      --> 99,441 rows
olist_geolocation_dataset                    --> 1,000,163 rows
olist_orders_dataset                         --> 99,441 rows
olist_order_items_dataset                    --> 112,650 rows
olist_order_payments_dataset                 --> 103,886 rows
olist_order_reviews_dataset                  --> 99,224 rows
olist_products_dataset                       --> 32,951 rows
olist_sellers_dataset                        --> 3,095 rows
product_category_name_translation            --> 71 rows


In [4]:
for table in table_names:
    print("-" *60)
    print(f"TABLE :{table}")
    df=pd.read_sql(f"select * FROM {table} LIMIT 3",conn)
    print(df)
    print()
    

------------------------------------------------------------
TABLE :olist_customers_dataset
                        customer_id                customer_unique_id  \
0  06b8999e2fba1a1fbc88172c00ba8bc7  861eff4711a542e4b93843c6dd7febb0   
1  18955e83d337fd6b2def6b18a428ac77  290c77bc529b7ac935b93aa66c333dc3   
2  4e7b3e00288586ebd08712fdd0374a03  060e732b5b29e8181a18229c7b0b2b5e   

   customer_zip_code_prefix          customer_city customer_state  
0                     14409                 franca             SP  
1                      9790  sao bernardo do campo             SP  
2                      1151              sao paulo             SP  

------------------------------------------------------------
TABLE :olist_geolocation_dataset
   geolocation_zip_code_prefix  geolocation_lat  geolocation_lng  \
0                         1037       -23.545621       -46.639292   
1                         1046       -23.546081       -46.644820   
2                         1046       -23.546

imp.

In [5]:
sellers=pd.read_sql("SELECT * FROM olist_sellers_dataset LIMIT 5",conn)
print(sellers)


                          seller_id  seller_zip_code_prefix  \
0  3442f8959a84dea7ee197c632cb2df15                   13023   
1  d1b65fc7debc3361ea86b5f14c68d2e2                   13844   
2  ce3ad9de960102d0677a81f5d0bb7b2d                   20031   
3  c0f3eea2e14555b6faeea3dd58c1b1c3                    4195   
4  51a04a8a6bdcb23deccc82b0b80742cf                   12914   

         seller_city seller_state  
0           campinas           SP  
1         mogi guacu           SP  
2     rio de janeiro           RJ  
3          sao paulo           SP  
4  braganca paulista           SP  


In [6]:
my_seller_id = 'd1b65fc7debc3361ea86b5f14c68d2e2'
print("-" * 60)
print("seller info: ")
print("-" * 60)
seller_info=pd.read_sql(f"""
    SELECT * FROM olist_sellers_dataset
    WHERE seller_id= '{my_seller_id}'
""",conn)
print(seller_info)
print("\n" + "_" * 60)
print("ORDERS BY THIS SELLER:")
print("\n" + "_" * 60)
seller_orders=pd.read_sql(f"""
    SELECT * FROM olist_order_items_dataset
    WHERE seller_id ='{my_seller_id}'
    """,conn)
print(f"total orders :{len(seller_orders)}")
print(seller_orders.head())
print("\n" + "_" * 60)
print("TOTAL REVENUE BY THIS SELLER:")
print("=" * 60)
seller_revenue=pd.read_sql(f"""
    SELECT
        seller_id,
        COUNT(order_id)  AS total_orders,
        ROUND(SUM(price),2) AS total_orders,
        ROUND(SUM(freight_value),2) AS total_freight
    FROM olist_order_items_dataset
    where seller_id ='{my_seller_id}'
    GROUP BY seller_id                                                                     
""",conn)
print(seller_revenue)




------------------------------------------------------------
seller info: 
------------------------------------------------------------
                          seller_id  seller_zip_code_prefix seller_city  \
0  d1b65fc7debc3361ea86b5f14c68d2e2                   13844  mogi guacu   

  seller_state  
0           SP  

____________________________________________________________
ORDERS BY THIS SELLER:

____________________________________________________________
total orders :41
                           order_id  order_item_id  \
0  035201c3c82a97f8a25dd6bd5244b2d5              1   
1  0504447548229e075dea8441b37b1e2a              1   
2  06e4e324b83309e575442446e34377d7              1   
3  16fd9b360cbadca08b1026c8532820f5              1   
4  1887b26eba419ce20f6fe34a6f88e80a              1   

                         product_id                         seller_id  \
0  1c36a5285f7f3b1ed2637d7c528ce5ff  d1b65fc7debc3361ea86b5f14c68d2e2   
1  8852eb03b04ec3268a66e9b696e25f6f  d1b65fc

In [7]:
seller_reviews = pd.read_sql(f"""
    SELECT
        r.review_score,
        COUNT(*) as total_reviews
    FROM olist_order_reviews_dataset r
    JOIN olist_order_items_dataset i
        ON r.order_id=i.order_id
    WHERE i.seller_id='{my_seller_id}'
    GROUP BY r.review_score
    ORDER BY r.review_score
""",conn)
print("reviews for the seller")
print(seller_reviews)

reviews for the seller
   review_score  total_reviews
0             1              2
1             3              1
2             4              8
3             5             30


In [8]:
# seller revenue & freight summary
revenue_summary=pd.read_sql("""
    SELECT
        i.seller_id,
        COUNT(DISTINCT i.order_id)      AS total_orders,
        ROUND(SUM(I.price),2)           AS total_revenue,
        ROUND(SUM(i.freight_value),2)   AS total_freight_cost,
        ROUND(AVG(i.price),2)           AS avg_order_value
    FROM olist_order_items_dataset i                 
    GROUP BY i.seller_id
""",conn)
print("revenue summary shape",revenue_summary.shape)
print(revenue_summary.head(3))

# seller review/rating summary
review_summary=pd.read_sql("""
    SELECT
        i.seller_id,
        ROUND(AVG(r.review_score),2)    AS avg_rating,
        COUNT(r.review_id)              AS total_reviews
    FROM olist_order_reviews_dataset r                           
    JOIN olist_order_items_dataset i
        ON r.order_id=i.order_id
    GROUP BY i.seller_id
""",conn)

print("review summary shape :",review_summary.shape)
print(review_summary.head(3))

# delivery delay summary 
delivery_summary =pd.read_sql("""
    SELECT
        i.seller_id,
        COUNT(CASE
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
            THEN 1 END)                 AS late_deliveries,
        COUNT(o.order_id)               AS total_delivered_orders,
        ROUND(
            COUNT(CASE
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date
                THEN 1 END) * 100.0 / COUNT(o.order_id)
        ,2)                                     AS late_delivery_pct          
    FROM olist_orders_dataset o
    JOIN olist_order_items_dataset i
        ON o.order_id = i.order_id
    WHERE o.order_status = 'delivered'
    GROUP BY i.seller_id
""",conn)
print("delivery summary shape :",delivery_summary.shape)
print(delivery_summary.head(3))


revenue summary shape (3095, 5)
                          seller_id  total_orders  total_revenue  \
0  0015a82c2db000af6aaaf3ae2ecb0532             3        2685.00   
1  001cca7ae9ae17fb1caed9dfb1094831           200       25080.03   
2  001e6ad469a905060d959994f1b41e4f             1         250.00   

   total_freight_cost  avg_order_value  
0               63.06           895.00  
1             8854.14           104.94  
2               17.94           250.00  
review summary shape : (3090, 3)
                          seller_id  avg_rating  total_reviews
0  0015a82c2db000af6aaaf3ae2ecb0532        3.67              3
1  001cca7ae9ae17fb1caed9dfb1094831        3.90            236
2  001e6ad469a905060d959994f1b41e4f        1.00              1
delivery summary shape : (2970, 4)
                          seller_id  late_deliveries  total_delivered_orders  \
0  0015a82c2db000af6aaaf3ae2ecb0532                0                       3   
1  001cca7ae9ae17fb1caed9dfb1094831               1

In [9]:
# top category per seller
category_summary = pd.read_sql("""
    SELECT 
        seller_id,
        top_category
    FROM (
        SELECT 
            i.seller_id,
            t.product_category_name_english  AS top_category,
            COUNT(*)                         AS category_count,
            ROW_NUMBER() OVER (
                PARTITION BY i.seller_id 
                ORDER BY COUNT(*) DESC
            )                                AS rank
        FROM olist_order_items_dataset i
        JOIN olist_products_dataset p
            ON i.product_id = p.product_id
        JOIN product_category_name_translation t
            ON p.product_category_name = t.product_category_name
        GROUP BY 
            i.seller_id,
            t.product_category_name_english
    )
    WHERE rank = 1
""", conn)

print("Category Summary Shape:", category_summary.shape)
print(category_summary.head(3))

Category Summary Shape: (3033, 2)
                          seller_id      top_category
0  0015a82c2db000af6aaaf3ae2ecb0532  small_appliances
1  001cca7ae9ae17fb1caed9dfb1094831      garden_tools
2  001e6ad469a905060d959994f1b41e4f    sports_leisure


In [10]:
# final table 
seller_final =pd.read_sql("""
    SELECT 
        s.seller_id,
        s.seller_city,
        s.seller_state,
                          
        -- revenue columns 
        COUNT(DISTINCT i.order_id)      AS total_orders,
        ROUND(SUM(i.price),2)           AS total_revenue,
        ROUND(SUM(i.freight_value),2)   AS total_freight_cost,
        ROUND(AVG(i.price),2)           AS avg_order_value,
    
        --review columns                 
        ROUND(AVG(r.review_score),2)    AS avg_rating,
        COUNT(r.review_id)              AS total_reviews,   

        -- Delivery columns
        COUNT(CASE 
            WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date 
            THEN 1 END)                         AS late_deliveries,
        ROUND(
            COUNT(CASE 
                WHEN o.order_delivered_customer_date > o.order_estimated_delivery_date 
                THEN 1 END) * 100.0 / COUNT(DISTINCT o.order_id)
        , 2)                                    AS late_delivery_pct,

        -- Category
        t.product_category_name_english         AS top_category

    FROM olist_sellers_dataset s
    JOIN olist_order_items_dataset i
        ON s.seller_id = i.seller_id
    JOIN olist_orders_dataset o
        ON i.order_id = o.order_id
    JOIN olist_order_reviews_dataset r
        ON o.order_id = r.order_id
    JOIN olist_products_dataset p
        ON i.product_id = p.product_id
    JOIN product_category_name_translation t
        ON p.product_category_name = t.product_category_name
    WHERE o.order_status = 'delivered'
    GROUP BY 
        s.seller_id,
        s.seller_city,
        s.seller_state,
        t.product_category_name_english

""", conn)

print(" Final Table Shape:", seller_final.shape)
print("\nColumns:", list(seller_final.columns))
print("\nSample Data:")
print(seller_final.head())                     



 Final Table Shape: (6123, 12)

Columns: ['seller_id', 'seller_city', 'seller_state', 'total_orders', 'total_revenue', 'total_freight_cost', 'avg_order_value', 'avg_rating', 'total_reviews', 'late_deliveries', 'late_delivery_pct', 'top_category']

Sample Data:
                          seller_id  seller_city seller_state  total_orders  \
0  0015a82c2db000af6aaaf3ae2ecb0532  santo andre           SP             3   
1  001cca7ae9ae17fb1caed9dfb1094831    cariacica           ES            23   
2  001cca7ae9ae17fb1caed9dfb1094831    cariacica           ES           169   
3  002100f778ceb8431b7a1020ff7ab48f       franca           SP            50   
4  004c9cd9d87a3c30c522c48c4fc07416     ibitinga           SP           143   

   total_revenue  total_freight_cost  avg_order_value  avg_rating  \
0        2685.00               63.06           895.00        3.67   
1        2925.73             1269.09           108.36        4.04   
2       21251.30             7290.53           104.17    

In [11]:
from sqlalchemy import create_engine

engine = create_engine('sqlite:///../data/olist_ecommerce.db')

seller_final.to_sql('seller_summary', con=engine, 
                     if_exists='replace', index=False)

print(f"seller_summary table saved!")
print(f"Total sellers: {seller_final['seller_id'].nunique()}")
print(f"Total rows: {len(seller_final):,}")

seller_summary table saved!
Total sellers: 2907
Total rows: 6,123


In [13]:
verify = pd.read_sql("SELECT * FROM seller_summary LIMIT 5", conn)
print(" Verified from database!")
print(verify)

 Verified from database!
                          seller_id  seller_city seller_state  total_orders  \
0  0015a82c2db000af6aaaf3ae2ecb0532  santo andre           SP             3   
1  001cca7ae9ae17fb1caed9dfb1094831    cariacica           ES            23   
2  001cca7ae9ae17fb1caed9dfb1094831    cariacica           ES           169   
3  002100f778ceb8431b7a1020ff7ab48f       franca           SP            50   
4  004c9cd9d87a3c30c522c48c4fc07416     ibitinga           SP           143   

   total_revenue  total_freight_cost  avg_order_value  avg_rating  \
0        2685.00               63.06           895.00        3.67   
1        2925.73             1269.09           108.36        4.04   
2       21251.30             7290.53           104.17        3.96   
3        1236.50              791.35            22.48        4.04   
4       18280.20             3085.46           114.25        4.15   

   total_reviews  late_deliveries  late_delivery_pct  \
0              3             

In [14]:
# fixing duplicate seller
seller_final = seller_final.groupby(
    ['seller_id', 'seller_city', 'seller_state'], 
    as_index=False
).agg({
    'total_orders'      : 'sum',
    'total_revenue'     : 'sum',
    'total_freight_cost': 'sum',
    'avg_order_value'   : 'mean',
    'avg_rating'        : 'mean',
    'total_reviews'     : 'sum',
    'late_deliveries'   : 'sum',
    'late_delivery_pct' : 'mean',
    'top_category'      : 'first'
})

print("Duplicates fixed!")
print("Shape after fix:", seller_final.shape)

Duplicates fixed!
Shape after fix: (2907, 12)


In [15]:
# adding new columns

# 1. Gross Profit — freight cost 
seller_final['gross_profit'] = (
    seller_final['total_revenue'] - 
    seller_final['total_freight_cost']
)

# 2. Profit Margin % 
seller_final['profit_margin'] = round(
    (seller_final['gross_profit'] / 
     seller_final['total_revenue']) * 100, 2
)

# 3. Revenue Per Order
seller_final['revenue_per_order'] = round(
    seller_final['total_revenue'] / 
    seller_final['total_orders'], 2
)

# 4. Seller Performance Label
def performance_label(row):
    if row['total_revenue'] >= 10000 and row['avg_rating'] >= 4.0:
        return 'Top Performer'
    elif row['total_revenue'] >= 5000 and row['avg_rating'] >= 3.5:
        return 'Mid Performer'
    else:
        return 'Low Performer'

seller_final['performance_label'] = seller_final.apply(
    performance_label, axis=1
)

# 5. Delivery Status Label
seller_final['delivery_status'] = seller_final['late_delivery_pct'].apply(
    lambda x: 'Good' if x <= 10 else ('Average' if x <= 25 else 'Poor')
)

print(" New feature columns added!")
print("\nAll Columns Now:")
for col in seller_final.columns:
    print(f"  - {col}")

print("\nSample:")
print(seller_final.head(3))

 New feature columns added!

All Columns Now:
  - seller_id
  - seller_city
  - seller_state
  - total_orders
  - total_revenue
  - total_freight_cost
  - avg_order_value
  - avg_rating
  - total_reviews
  - late_deliveries
  - late_delivery_pct
  - top_category
  - gross_profit
  - profit_margin
  - revenue_per_order
  - performance_label
  - delivery_status

Sample:
                          seller_id  seller_city seller_state  total_orders  \
0  0015a82c2db000af6aaaf3ae2ecb0532  santo andre           SP             3   
1  001cca7ae9ae17fb1caed9dfb1094831    cariacica           ES           192   
2  002100f778ceb8431b7a1020ff7ab48f       franca           SP            50   

   total_revenue  total_freight_cost  avg_order_value  avg_rating  \
0        2685.00               63.06          895.000        3.67   
1       24177.03             8559.62          106.265        4.00   
2        1236.50              791.35           22.480        4.04   

   total_reviews  late_deliveries  

In [16]:
# Updated table 
seller_final.to_sql('seller_summary', con=engine, 
                     if_exists='replace', index=False)

print(" Final seller_summary saved!")
print(f"Total Sellers: {len(seller_final):,}")
print(f"\nPerformance Distribution:")
print(seller_final['performance_label'].value_counts())
print(f"\nDelivery Status Distribution:")
print(seller_final['delivery_status'].value_counts())

 Final seller_summary saved!
Total Sellers: 2,907

Performance Distribution:
performance_label
Low Performer    2418
Mid Performer     296
Top Performer     193
Name: count, dtype: int64

Delivery Status Distribution:
delivery_status
Good       2202
Average     407
Poor        298
Name: count, dtype: int64
